# Credit Risk Prediction — Exploratory Data Analysis

This notebook walks through the dataset used to train the XGBoost loan default model.

**Run order:** Run all cells top-to-bottom after placing `loan_dataset.csv` in `data/raw/`.

In [ ]:
import sys
from pathlib import Path

# Make project root importable
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# FIX: original notebook used a hardcoded path 'loan_data.csv' which doesn't exist.
# Use config so the path is always consistent with the rest of the project.
from utils.config import RAW_DATA_PATH, TARGET_COLUMN

df = pd.read_csv(RAW_DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

## 1. Basic Info & Null Check

In [ ]:
print('Dtypes:')
print(df.dtypes.value_counts())
print()
null_pct = df.isnull().mean().sort_values(ascending=False)
print('Top-15 columns by null %:')
print(null_pct.head(15).to_string())

## 2. Target Distribution

In [ ]:
counts = df[TARGET_COLUMN].value_counts()
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(['Repay (0)', 'Default (1)'], counts.values, color=['#22c55e', '#ef4444'])
for i, v in enumerate(counts.values):
    ax.text(i, v + 100, str(v), ha='center', va='bottom', fontsize=9)
ax.set_title('Target Class Distribution')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()
print(f'Class balance: {counts[0]/(counts[0]+counts[1]):.1%} repay / {counts[1]/(counts[0]+counts[1]):.1%} default')

## 3. Numeric Feature Distributions

In [ ]:
NUM_FEATURES = ['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'fico_range_low',
                'installment', 'revol_bal', 'revol_util']

present = [c for c in NUM_FEATURES if c in df.columns]
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for ax, col in zip(axes.flat, present):
    df[col].dropna().plot(kind='hist', bins=40, ax=ax, color='#3b82f6', alpha=0.7, edgecolor='none')
    ax.set_title(col)
    ax.set_xlabel('')
plt.suptitle('Numeric Feature Histograms', y=1.01)
plt.tight_layout()
plt.show()

## 4. Default Rate by Grade

In [ ]:
if 'grade' in df.columns:
    dr = df.groupby('grade')[TARGET_COLUMN].mean().sort_index()
    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.bar(dr.index, dr.values * 100, color='#f59e0b')
    ax.set_ylabel('Default Rate (%)')
    ax.set_title('Default Rate by Loan Grade')
    for bar, val in zip(bars, dr.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val*100:.1f}%', ha='center', va='bottom', fontsize=8)
    plt.tight_layout()
    plt.show()

## 5. Correlation Heatmap

In [ ]:
corr_cols = [c for c in NUM_FEATURES + [TARGET_COLUMN] if c in df.columns]
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(corr_cols, fontsize=8)
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=7)
ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.show()

## 6. FICO Score vs Default Rate

In [ ]:
if 'fico_range_low' in df.columns:
    df['fico_bin'] = pd.cut(df['fico_range_low'], bins=range(300, 870, 20))
    fico_dr = df.groupby('fico_bin', observed=True)[TARGET_COLUMN].mean()
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(range(len(fico_dr)), fico_dr.values * 100, color='#ef4444', alpha=0.8)
    ax.set_xticks(range(len(fico_dr)))
    ax.set_xticklabels([str(b) for b in fico_dr.index], rotation=45, ha='right', fontsize=7)
    ax.set_ylabel('Default Rate (%)')
    ax.set_title('Default Rate by FICO Score Bucket')
    plt.tight_layout()
    plt.show()